In [93]:
import pandas as pd
import sqlite3
import warnings
import numpy as np
import joblib

from sklearn.compose import ColumnTransformer
from urllib.parse import urlsplit
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.neighbors import KNeighborsClassifier

from sklearn.model_selection import GridSearchCV
from sklearn.svm import SVC
from sklearn.ensemble import RandomForestClassifier, VotingClassifier, GradientBoostingClassifier
from catboost import CatBoostClassifier
from xgboost import XGBClassifier
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import (
    accuracy_score,
    classification_report,
    confusion_matrix
)

from sklearn.base import BaseEstimator, ClassifierMixin, clone
warnings.filterwarnings('ignore')

In [ ]:
conn = sqlite3.connect("../../db/telemetry.db")

In [ ]:
df = pd.read_sql_query(
    """
    SELECT
        event_id,
        title,
        executable,
        NULL AS url,
        publisher,
        event_start_time,
        event_end_time,
        previous_events,
        class,
        'OS' AS source
    FROM os_events
    WHERE executable NOT IN ('chrome.exe') AND class IS NOT NULL

    UNION ALL

    SELECT
        oe.event_id,
        be.title,
        oe.executable,
        be.url,
        oe.publisher,
        be.event_start_time,
        be.event_end_time,
        NULL AS previous_events,
        be.class,
        'BROWSER' AS source
    FROM browser_events AS be
    JOIN os_events AS oe
        ON be.os_event_id = oe.event_id
    WHERE be.class IS NOT NULL

    """,
    conn,
    parse_dates=["event_start_time", "event_end_time"]
)

df.head()

# Data Overview

In [ ]:
df[df["class"].isna()].head()

In [ ]:
df['class'].value_counts()

In [ ]:
df.info()

In [ ]:
df.describe()

In [ ]:
df['executable'].value_counts()

# Clean the data

In [ ]:
df = df[~df['title'].isna()]
df.info()

In [ ]:
df[~df['url'].isna()]['executable'].value_counts()


# Feature Engineering

### Constants

In [ ]:
BROWSER_EXECUTABLES = [
    'chrome.exe'
]

### Utility functions

In [ ]:

def get_time_information(df):
    df['week_day'] = df["event_start_time"].dt.weekday
    df['hour'] = df["event_start_time"].dt.hour
    df["duration_seconds"] = (
        df["event_end_time"] - df["event_start_time"]
    ).dt.total_seconds()

    return df

def parse_dates(df):
    df['event_start_time'] = pd.to_datetime(df['event_start_time'], format='%Y-%m-%d %H:%M:%S')
    df['event_end_time'] = pd.to_datetime(df['event_end_time'], format='%Y-%m-%d %H:%M:%S')
    return df


In [ ]:
df.head()

In [ ]:
df = parse_dates(df)
df = get_time_information(df)
df["domain"] = df['url'].apply(lambda url: urlsplit(url).hostname if type(url)!=float else url) 
df['domain'].value_counts()

In [ ]:
df.columns

In [ ]:
import json
previous_events_df = df[~df['previous_events'].isna()][['event_id', 'previous_events']]
previous_events_df['previous_events'] = previous_events_df['previous_events'].apply(json.loads)
previous_events_df = previous_events_df.explode('previous_events')

# Extracting the columns from the previous events 
previous_events_df['previous_executable'] = previous_events_df['previous_events'].apply(lambda x: x.get('executable') if type(x)!=float else x)
previous_events_df['previous_publisher'] = previous_events_df['previous_events'].apply(lambda x: x.get('publisher') if type(x)!=float else x)
previous_events_df['event_start_time'] = previous_events_df['previous_events'].apply(lambda x: x.get('event_start_time') if type(x)!=float else x)
previous_events_df['event_end_time'] = previous_events_df['previous_events'].apply(lambda x: x.get('event_end_time') if type(x)!=float else x)


previous_events_df = parse_dates(previous_events_df)
previous_events_df = get_time_information(previous_events_df)

# Aggregating the data
previous_events_df = (
    previous_events_df.groupby(
        'event_id'
    )
    .agg(
        previous_events_num_unique_executables=('previous_executable', 'nunique'),
        previous_events_num_unique_publishers=('previous_publisher', 'nunique'),
        previous_events_duration_seconds=('duration_seconds', 'sum'),
        
    )
    
)

previous_events_df.head()


In [ ]:
df['class'].value_counts()

In [ ]:
os_events_df = df[df['source']=='OS']
browser_events_df = df[df['source']=='BROWSER']

In [ ]:
os_events_df['class'].value_counts()

In [ ]:
previous_events_df.head()

In [ ]:
os_events_df = os_events_df.merge(previous_events_df, on='event_id', how='left')

In [ ]:
os_events_df.head()

In [ ]:
os_events_df['class'].value_counts()

In [ ]:
os_events_df['previous_events_parsed'] = os_events_df['previous_events'].apply(lambda value: json.loads(value) if type(value)!=float else value)
properties = ['executable', 'publisher']
for i in range(5):
    for property_ in properties:
        os_events_df[f'previous_event_{i + 1}_{property_}'] = os_events_df['previous_events_parsed'].apply(lambda arr: arr[i].get(property_) if type(arr)!=float and len(arr) > i else None)
os_events_df.head()


In [ ]:
os_events_df.info()

In [ ]:
df = pd.concat([os_events_df, browser_events_df])
df.info()

# Model Training

In [ ]:

text_features = ['title']
categorical_features = [
    'executable',
    'previous_event_1_executable',
    'previous_event_2_executable',
    'previous_event_3_executable',
    'previous_event_4_executable',
    'previous_event_5_executable',
    'previous_event_1_publisher',
    'previous_event_2_publisher',
    'previous_event_3_publisher',
    'previous_event_4_publisher',
    'previous_event_5_publisher',
    'week_day',
    'publisher',
    'domain',
]
numeric_features = [
    'hour', 
    'duration_seconds',
    'previous_events_duration_seconds',
    'previous_events_num_unique_executables',
    'previous_events_num_unique_publishers'
]

preprocessor = ColumnTransformer(
    transformers=[
        ('title', TfidfVectorizer(
            max_features=5000,
            ngram_range=(1, 2),
            min_df=2
        ), 'title'),
        ('categorical', OneHotEncoder(
            handle_unknown='ignore'
        ), categorical_features),
        ('numeric', Pipeline([
            ('imputer', SimpleImputer(strategy='median')),
            ('scaler', StandardScaler())
        ]), numeric_features)
    ]
)

model = Pipeline([
    ('preprocessor', preprocessor),
    ('classifier', LogisticRegression(max_iter=1000))
])

X = df[
    text_features + categorical_features + numeric_features
]
y = df['class']

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=31,
    stratify=y
)


## Single Test

In [ ]:
model.fit(X=X_train, y=y_train)


### A couple manual predictions:

In [ ]:
studying_prediction = pd.DataFrame([{
    'title': 'Visual Studio - Code',
    'executable': 'code.exe',
    'previous_event_1_executable': 'code.exe',
    'week_day': 'Saturday',
    'hour': 19,
    'duration_seconds': 300,
    'num_unique_urls': 0,
    'previous_event_3_publisher': "code.exe",
    'num_unique_domains': 0,
    'previous_event_2_executable': "code.exe",
    'previous_event_4_publisher': "Microsoft",
    'previous_event_5_publisher': "Microsoft",
    'previous_event_1_publisher': "Microsoft",
    'previous_event_2_publisher': "Microsoft",
    'num_browser_events': 0,
    'previous_event_5_executable': "code.exe",
    'previous_event_4_executable': "code.exe",
    'previous_event_3_executable': "code.exe",
    'publisher': 'Microsoft'
}])

In [ ]:
gaming_prediction = pd.DataFrame([{
    'title': 'Team Fortress 2 - Steam',
    'executable': 'tf2.exe',
    'previous_event_1_executable': 'steam.exe',
    'week_day': 'Saturday',
    'hour': 19,
    'duration_seconds': 300,
    'num_unique_urls': 0,
    'previous_event_3_publisher': "steam.exe",
    'num_unique_domains': 0,
    'previous_event_2_executable': "steam.exe",
    'previous_event_4_publisher': "Valve Corporation",
    'previous_event_5_publisher': "Valve Corporation",
    'previous_event_1_publisher': "Valve Corporation",
    'previous_event_2_publisher': "Valve Corporation",
    'num_browser_events': 0,
    'previous_event_5_executable': "steam.exe",
    'previous_event_4_executable': "steam.exe",
    'previous_event_3_executable': "steam.exe",
    'publisher': 'Valve Corporation'
}])

In [ ]:
entertainment_class = pd.DataFrame([{
    'title': 'Chrome - Youtube',
    'executable': 'chrome.exe',
    'previous_event_1_executable': 'chrome.exe',
    'week_day': 'Saturday',
    'hour': 19,
    'duration_seconds': 300,
    'num_unique_urls': 0,
    'previous_event_3_publisher': "chrome.exe",
    'num_unique_domains': 0,
    'previous_event_2_executable': "chrome.exe",
    'previous_event_4_publisher': "Google LLC",
    'previous_event_5_publisher': "Google LLC",
    'previous_event_1_publisher': "Google LLC",
    'previous_event_2_publisher': "Google LLC",
    'num_browser_events': 0,
    'previous_event_5_executable': "chrome.exe",
    'previous_event_4_executable': "chrome.exe",
    'previous_event_3_executable': "chrome.exe",
    'publisher': 'Google LLC'
}])

In [ ]:
other_class = pd.DataFrame([{
    'title': 'Rozetka - Щоразу, що треба!',
    'executable': 'chrome.exe',
    'previous_event_1_executable': 'chrome.exe',
    'week_day': 'Saturday',
    'hour': 19,
    'duration_seconds': 300,
    'num_unique_urls': 0,
    'previous_event_3_publisher': "chrome.exe",
    'num_unique_domains': 0,
    'previous_event_2_executable': "chrome.exe",
    'previous_event_4_publisher': "Google LLC",
    'previous_event_5_publisher': "Google LLC",
    'previous_event_1_publisher': "Google LLC",
    'previous_event_2_publisher': "Google LLC",
    'num_browser_events': 0,
    'previous_event_5_executable': "chrome.exe",
    'previous_event_4_executable': "chrome.exe",
    'previous_event_3_executable': "chrome.exe",
    'publisher': 'Google LLC'
}])

In [ ]:
social_media_input = pd.DataFrame([{
    'title': 'Telegram',
    'executable': 'telegram.exe',
    'previous_event_1_executable': 'telegram.exe',
    'week_day': 'Saturday',
    'hour': 19,
    'duration_seconds': 300,
    'num_unique_urls': 0,
    'previous_event_3_publisher': "telegram.exe",
    'num_unique_domains': 0,
    'previous_event_2_executable': "telegram.exe",
    'previous_event_4_publisher': "Telegram FZ-LLC",
    'previous_event_5_publisher': "Telegram FZ-LLC",
    'previous_event_1_publisher': "Telegram FZ-LLC",
    'previous_event_2_publisher': "Telegram FZ-LLC",
    'num_browser_events': 0,
    'previous_event_5_executable': "telegram.exe",
    'previous_event_4_executable': "telegram.exe",
    'previous_event_3_executable': "telegram.exe",
    'publisher': 'Telegram FZ-LLC',
    'previous_events_num_unique_executables': 3, 
    'previous_events_duration_seconds': 20,
    'previous_events_num_unique_publishers': 3
}])

In [ ]:
prediction = model.predict(social_media_input)

print(prediction[0])

In [ ]:
y_pred = model.predict(X_test)
# Accuracy
print("Accuracy:", accuracy_score(y_test, y_pred))
print(classification_report(y_test, y_pred))

## Model Comparison

In [ ]:
models = [
    LogisticRegression(max_iter=1000),
    SVC(
        kernel="rbf",
        C=10.0,
        random_state=13
    ),
    RandomForestClassifier(
        n_estimators=100,
        random_state=13,
        n_jobs=-1
    )
]


for model in models:
    pipeline = Pipeline([
        ('preprocessor', preprocessor),
        ('classifier', model)
    ])
    pipeline.fit(X=X_train, y=y_train)
    
    y_pred = pipeline.predict(X_test)
    
    print(f"Accuracy for {model}:", accuracy_score(y_test, y_pred))
    print(classification_report(y_test, y_pred))

In [ ]:
def perform_grid_search(y_train):
    for name, config in models.items():
        pipeline = Pipeline([
            ('preprocessor', preprocessor),
            ('model', config['model'])
        ])

        grid = GridSearchCV(
            estimator=pipeline,
            param_grid=config["params"],
            cv=5,
            scoring="f1_macro",
            n_jobs=-1
        )

        grid.fit(X_train, y_train)

        print(f"{name}: {grid.best_score_:.4f}")
        print(f"Best parameters: {grid.best_params_}\n")

In [ ]:
models = {
    "LogisticRegression": {
        "model": LogisticRegression(max_iter=1000),
        "params": {
            "model__C": [0.1, 1, 10, 50, 100]
        }
    },
    "RandomForest": {
        "model": RandomForestClassifier(random_state=42),
        "params": {
            "model__n_estimators": [100, 200],
            "model__max_depth": [None, 10, 20]
        }
    },
    "SVC": {
        "model": SVC(),
        "params": {
            "model__C": [0.1, 1, 10, 50, 100],
            "model__kernel": ["linear", "rbf"]
        }
    },
    'KNN': {
        "model": KNeighborsClassifier(),
        "params": {
            "model__n_neighbors": [3, 5, 9, 17]
        }
    },
    'GradientBoostingClassifier': {
        "model": GradientBoostingClassifier(),
        "params": {
            "model__n_estimators": [50, 100, 200],
            "model__learning_rate": [0.01, 0.05, 0.1],
            "model__max_depth": [2, 3, 5],
            "model__subsample": [0.8, 1.0]
        }
    },
}

perform_grid_search(y_train=y_train)

In [ ]:
models = {
    'CatBoostClassifier': {
        "model": CatBoostClassifier(random_state=13),
        "params": {
            "model__iterations": [300, 500, 800],
            "model__learning_rate": [0.03, 0.05, 0.1],
            "model__depth": [4, 6, 8]
        }
    },
    'XGBClassifier': {
        "model": XGBClassifier(
            random_state=13,
            eval_metric="mlogloss",
            n_jobs=1
        ),
        "params": {
            "model__n_estimators": [100, 300],
            "model__max_depth": [3, 6],
            "model__learning_rate": [0.03, 0.1],
            "model__subsample": [0.8, 1.0],
            "model__colsample_bytree": [0.8, 1.0],
        }
    }

}

encoder = LabelEncoder()
y_train_encoded = encoder.fit_transform(y_train)
perform_grid_search(y_train=y_train_encoded)


In [ ]:
class CatBoostWrapper(ClassifierMixin, BaseEstimator):
    def __init__(self, model):
        self.model = model

    def fit(self, X, y):
        self.model_ = clone(self.model)
        self.model_.fit(X, np.asarray(y).ravel())
        self.classes_ = self.model_.classes_
        return self

    def predict(self, X):
        return np.asarray(self.model_.predict(X)).ravel()

    def predict_proba(self, X):
        return self.model_.predict_proba(X)

### Regular Classifiers

In [89]:
selected_models = [
    (
    'LogisticRegression',
        LogisticRegression(
            C=10
        ),
    ),
    (
    'SVC',
        SVC(
            C=10, kernel='linear', probability=True
        )
    ),
    (
    'RandomForestClassifier',
        RandomForestClassifier(
            max_depth=None, n_estimators=100
        )
    )
]

ensemble_classifier = VotingClassifier(estimators=selected_models, voting='hard')
pipeline = Pipeline([
    ('preprocessor', preprocessor),
    ('classifier', ensemble_classifier)
])
pipeline.fit(X_train, y_train)
y_pred = pipeline.predict(X_test)
# Accuracy
print("Accuracy:", accuracy_score(y_test, y_pred))
print(classification_report(y_test, y_pred))


Accuracy: 0.9961977186311787
               precision    recall  f1-score   support

entertainment       1.00      1.00      1.00        50
       gaming       1.00      1.00      1.00        14
        other       1.00      1.00      1.00        12
 social_media       0.98      1.00      0.99        49
     studying       1.00      0.99      1.00       138

     accuracy                           1.00       263
    macro avg       1.00      1.00      1.00       263
 weighted avg       1.00      1.00      1.00       263



#### Boosting Classifiers

In [ ]:
selected_models = [
    (
    'GradientBoostingClassifier',
        GradientBoostingClassifier(
            learning_rate=0.1, max_depth=2, n_estimators=200, subsample=0.8
        ), 
    ),
    (
    'CatBoostClassifier',
        CatBoostWrapper(CatBoostClassifier(
            depth=8, iterations=10, learning_rate=0.1
        )),
    ),
    (
    'XGBClassifier',
        XGBClassifier(
            colsample_bytree=0.8, learning_rate=0.03, max_depth=6, n_estimators=300, subsample=1.0
        )
    )
]

ensemble_classifier = VotingClassifier(estimators=selected_models, voting='hard')
pipeline = Pipeline([
    ('preprocessor', preprocessor),
    ('classifier', ensemble_classifier)
])
pipeline.fit(X_train, y_train)
y_pred = pipeline.predict(X_test)
# Accuracy
print("Accuracy:", accuracy_score(y_test, y_pred))
print(classification_report(y_test, y_pred))

### Final Estimator

In [92]:
selected_models = [
    (
    'LogisticRegression',
        LogisticRegression(
            C=10
        ),
    ),
    (
    'SVC',
        SVC(
            C=10, kernel='linear', probability=True
        )
    ),
    (
    'RandomForestClassifier',
        RandomForestClassifier(
            max_depth=None, n_estimators=100
        )
    )
]

ensemble_classifier = VotingClassifier(estimators=selected_models, voting='soft')
pipeline = Pipeline([
    ('preprocessor', preprocessor),
    ('classifier', ensemble_classifier)
])
pipeline.fit(X_train, y_train)
y_pred = pipeline.predict_proba(X_test)

print(pipeline.classes_)
most_likely_prediction_probability = np.max(y_pred, axis=1)
most_likely_prediction_class = pipeline.classes_[np.argmax(y_pred, axis=1)]
most_likely_prediction_class

['entertainment' 'gaming' 'other' 'social_media' 'studying']


array(['social_media', 'entertainment', 'entertainment', 'studying',
       'social_media', 'other', 'studying', 'studying', 'studying',
       'entertainment', 'entertainment', 'entertainment', 'entertainment',
       'gaming', 'studying', 'studying', 'studying', 'studying',
       'social_media', 'entertainment', 'gaming', 'other', 'studying',
       'social_media', 'studying', 'social_media', 'studying',
       'social_media', 'studying', 'entertainment', 'studying',
       'studying', 'social_media', 'studying', 'studying',
       'entertainment', 'gaming', 'studying', 'gaming', 'social_media',
       'studying', 'other', 'studying', 'entertainment', 'studying',
       'studying', 'entertainment', 'studying', 'studying', 'studying',
       'entertainment', 'studying', 'entertainment', 'studying',
       'studying', 'social_media', 'studying', 'entertainment',
       'studying', 'social_media', 'studying', 'social_media', 'studying',
       'studying', 'studying', 'studying', 'other

In [94]:
joblib.dump(pipeline, "event_classifier.joblib")

['event_classifier.joblib']